# VCG Training and Resource Results

This notebook reads the completed VCG loss-selection results. It does not train VCGs.

In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import display, Image

candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "code-tuning"]
REPO = next(path for path in candidates if (path / "final_experiment/config.json").exists())
os.environ.setdefault("MPLCONFIGDIR", str(REPO / "final_experiment/.cache/matplotlib"))
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "tuning"))
from analyze import read_results, status_summary, ranking, FigureWriter, figures
from resource_analysis import estimate_study_a, resource_figures

RESULTS_DIR = Path(os.environ.get(
    "PCQAOA_VCG_RESULTS", REPO / "final_experiment/vcg_results"))
if not (RESULTS_DIR / "run.json").exists():
    RESULTS_DIR = REPO / "tuning/results/full"
B_RESULTS_DIR = REPO / "tuning/results/full-b-five-methods"
data = read_results(RESULTS_DIR, B_RESULTS_DIR)
writer = FigureWriter(REPO / "final_experiment/results/vcg_figures", latex=True)
writers = {"a": writer}

def show_section(section, study="a"):
    for figure_path in figures(data, section, writer, study="a"):
        display(Image(filename=str(figure_path), width=900))
    print("Fonts:", writer.font_status)

print("VCG Results:", RESULTS_DIR)
print("Figure Directory:", writer.directory)

## Progress and Dataset Composition

Rerun the next cell to refresh this study. A completed VCG task can have fidelity below 0.999.

In [ ]:
data = read_results(RESULTS_DIR, B_RESULTS_DIR)
print("Results:", RESULTS_DIR)
print(json.dumps(status_summary(RESULTS_DIR)["a"], indent=2))
instances = data["a_instances"]
if not instances.empty:
    display(instances[['id', 'support', 'family', 'feasible_count', 'feasible_fraction']])
show_section("datasets", "a")

## Loss Functions and Fidelity

Let $\mathcal F$ be the feasible set and $|\psi_\theta\rangle$ the trained state. The target is

$$|u_{\mathcal F}\rangle=\frac{1}{\sqrt{|\mathcal F|}}\sum_{x\in\mathcal F}|x\rangle.$$

The three state measurements are

$$P_{\mathcal F}=\sum_{x\in\mathcal F}|\langle x|\psi_\theta\rangle|^2,\qquad
F=|\langle u_{\mathcal F}|\psi_\theta\rangle|^2,\qquad
F_{\mathrm{cond}}=\frac{F}{P_{\mathcal F}}.$$

The code sets $F_{\mathrm{cond}}=0$ when $P_{\mathcal F}\leq10^{-12}$ to avoid division by zero.

The run compares **five loss conditions**, using three formulas:

$$\begin{aligned}
L_{\mathrm{feas}} &= 1-2P_{\mathcal F},\\
L_{\mathrm{fid}} &= 1-F,\\
L_{0.5} &= (1-P_{\mathcal F})+0.5(1-F_{\mathrm{cond}}),\\
L_1 &= (1-P_{\mathcal F})+(1-F_{\mathrm{cond}}),\\
L_2 &= (1-P_{\mathcal F})+2(1-F_{\mathrm{cond}}).
\end{aligned}$$

At each depth, retain the restart with the lowest final training loss. Stop when its fidelity reaches 0.999 or after depth 8. Return the highest-fidelity state among the retained depths. Select the loss with the highest mean returned fidelity across all 50 constraints.

Depth plots include only tasks that reached each depth. Early stopping can change which constraints contribute to those means. Rankings shown before all tasks finish are incomplete.

In [ ]:
if not data["a"].empty:
    display(ranking(data["a"], "fidelity"))
    display(data["a"][["instance_id", "condition", "fidelity", "p_feasible", "conditional_fidelity",
                       "entropy", "coverage", "min_conditional_probability", "selected_depth", "converged"]])
show_section("loss", "a")

## Example States and Optimization Histories

Each distribution title shows the constraint and training loss. Blue bars represent feasible states; rose bars represent infeasible states. The examples have low, middle and high returned fidelity.

Training histories record loss before each indicated update. Saved final metrics are calculated after the final update.

The training-history plot shows one constraint at one ma-QAOA depth, identified in the title along with the loss function. Each curve is a separate optimizer restart. Color and line style identify the restart in the legend. The thicker curve marked **Retained** is the restart with the lowest final training loss at that depth. It need not be the restart with the highest fidelity. The horizontal axis counts Adam updates; the vertical axis is the training loss, not fidelity. Only saved history points are plotted.


In [ ]:
show_section("examples", "a")

## Constraint Resource Estimates

We count the gates needed to prepare each saved VCG. The counts use its selected depth and trained angles. No VCG is retrained.

- **NISQ** means noisy intermediate-scale quantum. Here, gates are counted using $\{R_X,R_Y,R_Z,H,\mathrm{CNOT},X,Y,Z,T,S\}$.
- **FTQC** means fault-tolerant quantum computing. Here, rotations are approximated using $\{H,T,\mathrm{CNOT},S\}$. These counts exclude error correction and physical hardware costs.
- The first two figures compare CNOT counts and T counts across the five losses.
- The next two compare **one selected VCG preparation** with **one penalty cost layer**. Repeated penalty terms are combined before counting. These are not full PC-QAOA versus PenaltyQAOA comparisons: Grover mixers also undo and repeat gadget preparation in every layer.
- `n_allocated` counts decision and slack qubits. Extra qubits used by the estimator are listed separately.

Individual gate counts are integers. Tables and curves show averages, which can have decimals. Error bars are 95% confidence intervals across constraints.

Counts use PennyLane's gate decompositions, without removing zero-angle gates. FTQC rotation precision is recorded in the saved estimates. Results are cached in `resource_estimates/` and recalculated when their inputs or estimator settings change.

In [ ]:
resource_a = estimate_study_a(RESULTS_DIR)
selection_path = RESULTS_DIR / "selected_loss.json"
selected_loss = json.loads(selection_path.read_text())["condition"] if selection_path.exists() else "fidelity"
print("Resource Rows:", len(resource_a))
if not resource_a.empty:
    display(resource_a.groupby(["representation", "status", "gate_set"]).size().rename("Rows"))
    display(resource_a.groupby(["gate_set", "condition", "representation"])[
        ["total_gates", "two_qubit_gates", "t_gates", "n_allocated", "total_wires"]].mean())
    display(resource_a[["instance_id", "condition", "representation", "gate_set",
        "selected_depth", "total_gates", "two_qubit_gates", "t_gates", "n_slack",
        "n_allocated", "algo_wires", "zeroed_wires", "any_state_wires"]])
    for path in resource_figures(resource_a, "a", writer, selected_loss):
        display(Image(filename=str(path), width=900))
    print("Fonts:", writer.font_status)


## Runtime

Compilation and optimizer-update times are recorded separately. Peak RSS is the largest resident memory use of the worker process.

In [ ]:
frame = data["a"]
if not frame.empty:
    display(frame.groupby("condition")[["compile_seconds", "optimize_seconds", "process_peak_rss_mb"]].agg(["mean", "std"]))

## Saved Selection and Reproducibility

The selection file is written after all tasks in this study finish.

In [ ]:
path = RESULTS_DIR / "selected_loss.json"
print(path.read_text() if path.exists() else "No final selection yet.")
print("Environment:")
print(json.dumps(data["run"]["environment"], indent=2))
print("Figure Directory:", writer.directory)